# Lab 02 – Random variables, the Poisson process, and queueing systems

Random samples from the uniform, exponential, and normal distributions, followed by a discrete-event simulation of packet sources, a switch, and sinks built on SimPy. The measured delay and occupancy of an M/M/1 system are compared with the analytical formulas, and the mean delay is traced as the utilization approaches one. Theory and task descriptions are in the [README](../README.md).

## Imports

In [ ]:
from functools import partial

import numpy as np
import panel as pn
import simpy

from lib.core import NetworkTap, Packet, PacketFork, PacketSink, PacketSource, Switch
from lib.packets import packet_table, packet_timeline
from lib.params import colors
from lib.plots import GREY, card, cards, histogram, legend_below, new_plot, sample_card
from lib.queueview import queue_view
from lib.report import (
    mm1_measurements,
    mm1_report,
    port_measurements,
    port_report,
    seed_table,
)
from lib.tapview import tap_card, tap_view

pn.extension("tabulator")  # the packet table needs the Tabulator widget

## Constants

In [ ]:
SEED = 1  # record this value; repeat the M/M/1 comparison with seeds 2 and 3
SIZE = 10_000  # samples per histogram

## Utility functions

The tables, cards, and the M/M/1 comparison are in `lib/report.py`; only the helper that starts a reproducible run belongs in the notebook itself.

In [ ]:
def new_run(seed=SEED):
    """
    Start a fresh simulation.

    Recreating the environment and the generator from the same seed, and
    restarting the packet numbering, makes a cell produce the same result every
    time it is run. Without it, a re-run continues the generator where the
    previous one stopped and the numbers change.

    Parameters
    ----------
    seed : int, optional
        Seed of the random number generator, by default SEED.

    Returns
    -------
    tuple[simpy.Environment, numpy.random.Generator]
        A new simulation environment and a new generator.
    """
    Packet.reset_counter()
    return simpy.Environment(), np.random.default_rng(seed)

## Theory

Each cell below reseeds the generator from `SEED`, draws `SIZE` samples with a [`numpy.random.Generator`](https://numpy.org/doc/stable/reference/random/generator.html) method, compares the sample moments with the theoretical ones, and plots a normalized histogram with the theoretical probability density function overlaid.

### Uniform distribution

In [ ]:
rng = np.random.default_rng(SEED)
x = rng.random(size=SIZE)

plot = new_plot(
    "Uniform distribution on [0, 1)",
    "Value [–]",
    "Probability density [–]",
    y_range=(0, 1.4),
)
histogram(plot, x, bins=50, label="Samples")
plot.line(
    [0, 1],
    [1, 1],
    line_color=GREY,
    line_dash="dashed",
    line_width=2,
    legend_label="Density f(x)",
)
legend_below(plot)

pn.Row(plot, sample_card(x, mean=0.5, std=np.sqrt(1 / 12)))

### Exponential distribution

In [ ]:
rng = np.random.default_rng(SEED)
scale = 1.0  # mean waiting time 1/λ [s]
x = rng.exponential(scale=scale, size=SIZE)

t = np.linspace(0, 6 * scale, 300)
plot = new_plot(
    "Exponential distribution with λ = 1 s⁻¹",
    "Inter-arrival time [s]",
    "Probability density [s⁻¹]",
    x_range=(0, 6 * scale),
)
histogram(plot, x, bins=60, label="Samples")
plot.line(
    t,
    np.exp(-t / scale) / scale,
    line_color=GREY,
    line_dash="dashed",
    line_width=2,
    legend_label="Density λ·exp(−λx)",
)
legend_below(plot)

pn.Row(plot, sample_card(x, mean=scale, std=scale))

### Normal distribution

In [ ]:
rng = np.random.default_rng(SEED)
mu_n, sigma_n = 0.0, 1.0
x = rng.normal(loc=mu_n, scale=sigma_n, size=SIZE)

t = np.linspace(-4 * sigma_n, 4 * sigma_n, 300)
plot = new_plot(
    "Normal distribution with μ = 0, σ = 1", "Value [–]", "Probability density [–]"
)
histogram(plot, x, bins=60, label="Samples")
plot.line(
    t,
    np.exp(-((t - mu_n) ** 2) / (2 * sigma_n**2)) / (sigma_n * np.sqrt(2 * np.pi)),
    line_color=GREY,
    line_dash="dashed",
    line_width=2,
    legend_label="Density f(x)",
)
legend_below(plot)

pn.Row(plot, sample_card(x, mean=mu_n, std=sigma_n))

### From exponential gaps to Poisson counts

The two descriptions of a homogeneous Poisson process are equivalent: drawing each gap between arrivals from an exponential distribution with rate $\lambda$ makes the number of arrivals in an interval of length $t$ Poisson distributed with mean $\lambda t$. The cell below draws the gaps, counts the arrivals in each one-second interval, and compares the counts with the Poisson probability mass function. A Poisson distribution has equal mean and variance, which is the quickest check to apply.

In [ ]:
rng = np.random.default_rng(SEED)
LAMBDA = 2.0  # arrival rate [s⁻¹]

gaps = rng.exponential(1 / LAMBDA, size=SIZE)
arrivals = np.cumsum(gaps)
counts = np.bincount(arrivals.astype(int))[:-1]  # drop the incomplete last second

k = np.arange(counts.max() + 1)
# log of the PMF, evaluated as k log λ − λ − log k!, to avoid a large factorial
pmf = np.exp(k * np.log(LAMBDA) - LAMBDA - np.cumsum(np.log(np.maximum(k, 1))))

plot = new_plot(
    "Exponential gaps produce Poisson counts",
    "Arrivals per interval [–]",
    "Probability [–]",
)
histogram(
    plot, counts, bins=np.arange(-0.5, counts.max() + 1.5), label="Simulated counts"
)
plot.line(
    k,
    pmf,
    line_color=GREY,
    line_dash="dashed",
    line_width=2,
    legend_label="Poisson PMF",
)
plot.scatter(k, pmf, size=9, color=GREY, legend_label="Poisson PMF")
plot.xaxis.ticker = list(k)
legend_below(plot)

pn.Row(
    plot,
    card(
        "Counts per 1 s against Poisson theory",
        [
            ("Intervals observed [–]", f"{counts.size}"),
            ("Mean [–]", f"{counts.mean():.3f} &nbsp;(theory {LAMBDA:.3f})"),
            ("Variance [–]", f"{counts.var():.3f} &nbsp;(theory {LAMBDA:.3f})"),
        ],
        note="A Poisson count has its variance equal to its mean, both λ times "
        "the length of the interval.",
    ),
)

## Tasks

### Step 1 – Source and sink

The simplest simulation: a `PacketSource` sends directly to a `PacketSink`. One simulation time unit (STU) represents one second in this lab.

After the run, `packet_table(sink)` lists every packet the sink received with its size, creation and arrival time, and time in the system. Click a column heading to sort by it, and type into the box beneath a heading to filter: a source name, or a number in *Time in system* to keep only the packets that spent at least that long. `packet_timeline(sink)` draws each packet as a bar from its creation to its arrival; hover over a bar for its details.

Passing `debug=True` to the sink, or to a source, also prints one line per packet as the simulation runs. That is handy for a run of a few packets and unreadable for a long one.

In [ ]:
env, _ = new_run()

sink = PacketSink(env, "sink")
source = PacketSource(
    env,
    "source01",
    packet_interval=1,  # interval between packets [STU]
    packet_size=10,  # packet size [B]
)
source.destination = sink

env.run(until=20)

pn.Column(packet_table(sink), packet_timeline(sink))

### Task 1 – Two sources with random traffic

1. Extend the simulation above to two sources with different packet sizes and inter-arrival times.
2. Replace the constants by distributions: `packet_interval` and `packet_size` accept any zero-argument callable, for example `partial(rng.exponential, 2)`. Start the cell with `env, rng = new_run()` so that the generator exists and the run stays reproducible.
3. Compare the packet table and the timeline with the constant-interval case. Filter the *Source* column to look at one source at a time.

### Step 2 – Source, switch, and sink

A `Switch` with four ports is inserted between source and sink. Packets arrive with uniformly distributed intervals from 1 to 3 s and exponentially distributed sizes; the port buffers 100 B of waiting packets and drains at 1000 bit/s.

In [ ]:
env, rng = new_run()

sink = PacketSink(env, "sink")
switch = Switch(
    env,
    "switch01",
    num_ports=4,  # each port is bidirectional, i.e. two physical ports of a real switch
    port_capacity=100,  # buffer for waiting packets [B]
    port_transmission_rate=1000,  # [bit/s]
)
source = PacketSource(
    env,
    "source01",
    packet_interval=partial(rng.uniform, 1, 3),
    packet_size=partial(rng.exponential, 50),
)

source.destination = switch.ports[0]
switch.ports[0].destination = sink

env.run(until=200)

# Passing the port lists the packets it dropped as well, in grey.
pn.Column(
    packet_table(sink, ports=switch.ports[0]),
    packet_timeline(sink, ports=switch.ports[0]),
)

### Task 2 – Time in the system

1. From the packet table of Step 2, find how long the packets spent in the system (sort by *Time in system*) and which packets the port dropped. They are listed in grey and marked with red crosses on the timeline; `switch.ports[0].cum_drop_count` counts them.
2. A packet of the mean size, 50 B, needs $8 \cdot 50 / 1000 = 0.4$ s to transmit, yet the mean time in system in the summary of the Step 2 packet table is smaller than that. Explain the discrepancy. What is the largest packet a 100 B buffer can ever accept, and how does that bias the sizes of the packets that reach the sink?
3. Reduce the transmission rate or the buffer size and observe how the delay and the drops change.

### Step 3 – Network tap

The sink records only the packets that made it through. A `NetworkTap` samples the port itself at a fixed interval, here every 0.05 s, and records what the sink cannot see: `queue_bytes`, the bytes waiting in the buffer; `queue_packets`, the packets waiting; and `system_packets`, the waiting packets plus the one being transmitted. At the load of Step 2 no packet ever waits: packets arrive at least 1 s apart, and a packet that needs longer than that to transmit is larger than 125 B, which the 100 B buffer never admits. A tap there would show an empty buffer throughout. Step 3 therefore keeps the port but offers it more traffic: exponentially distributed gaps with a mean of 0.5 s, the Poisson arrivals of Step 4.

The upper chart puts the bytes in the buffer next to its 100 B capacity and marks every dropped packet with a red cross at the level the buffer would have reached had the packet been admitted: the bytes already waiting plus the packet itself. Every cross therefore lies above the capacity line, some even where the buffer was empty, because the packet alone was larger than the buffer. Hover over a cross to see the numbers and the reason for the drop.

The lower chart stacks the packets at the port: those waiting in the buffer at the bottom, and the packet being transmitted, at most one, on top. The lower area alone is what *L*<sub>q</sub> counts, the whole stack what *L* counts; that is the difference between the two formulas of Step 4. The card beside the first one turns the samples into estimates of the utilization *ρ*, of *L*, and of *L*<sub>q</sub>. The charts open on the 30 s with the most drops; drag or zoom to see the rest of the run.

In [ ]:
env, rng = new_run()

sink = PacketSink(env, "sink")
switch = Switch(
    env,
    "switch01",
    num_ports=4,
    port_capacity=100,  # buffer for waiting packets [B]
    port_transmission_rate=1000,  # [bit/s]
)
port = switch.ports[0]
tap = NetworkTap(env, port=port, interval=0.05)  # one sample every 50 ms [s]
source = PacketSource(
    env,
    "source01",
    packet_interval=partial(rng.exponential, 0.5),  # Poisson arrivals, 2 per second
    packet_size=partial(rng.exponential, 50),
)

source.destination = port
port.destination = sink

env.run(until=200)

pn.Column(
    cards(
        card(
            "Packets at the port",
            [
                ("Offered [–]", port.cum_packet_count),
                ("Dropped [–]", port.cum_drop_count),
                ("Delivered [–]", len(sink.delays)),
                ("Mean time in system [s]", f"{np.mean(sink.delays):.3f}"),
            ],
            note="What the port counters and the sink report. Neither says how full "
            "the buffer was, or when.",
        ),
        tap_card(tap, port),
    ),
    tap_view(tap, port),
)

### Task 3 – How often to sample

1. Repeat the simulation of Step 3 with two taps on the same port, `coarse` sampling every 1 s and `fine` every 0.05 s. Several taps can sample one port without changing the simulation.
2. Show both with `cards(tap_card(coarse, port), tap_card(fine, port))`, and draw `tap_view(coarse, port)` below `tap_view(fine, port)`.
3. Which numbers agree and which do not? Explain why the busy share and the mean occupancies agree within the sampling noise, while the coarse tap sees lower peaks and misses many of the drops before they happen.
4. Operators often poll router counters only every few minutes. Which congestion events can such monitoring miss, and what would you measure instead?

### Step 4 – M/M/1 approximation

Exponential inter-arrival times with mean 2 STU, exponential packet sizes with mean 100 B, one port at 1000 bit/s with a 10 000 B buffer: a Poisson flow served by a single channel with exponentially distributed service times.

The finite buffer and the rounding of packet sizes to whole bytes make this an approximation of the ideal infinite-buffer model. The first `WARMUP` seconds are excluded, because the system starts empty and needs time to reach typical operating conditions.

In [ ]:
WARMUP = 1000  # discarded transient [s]
DURATION = 8000  # total simulated time [s]
MEAN_INTERARRIVAL = 2  # mean time between arrivals [STU]
MEAN_SIZE = 100  # mean packet size [B]
PORT_RATE = 1000  # link rate [bit/s]
BUFFER = 10_000  # buffer for waiting packets [B]

env, rng = new_run()

sink = PacketSink(env, "sink", debug=False)
switch = Switch(
    env,
    "switch01",
    num_ports=1,
    port_capacity=BUFFER,
    port_transmission_rate=PORT_RATE,
)
port = switch.ports[0]
tap = NetworkTap(env, port=port)
source = PacketSource(
    env,
    "source",
    packet_interval=partial(rng.exponential, MEAN_INTERARRIVAL),
    packet_size=partial(rng.exponential, MEAN_SIZE),
)

source.destination = port
port.destination = sink

env.run(until=DURATION)

In [ ]:
# Pair every theoretical symbol with the measurement that should confirm it.
# mm1_measurements returns the seven quantities keyed exactly like the theory
# dictionary of Task 4, plus the drop counters, and the arrays behind them.
measured, samples = mm1_measurements(
    sink, tap, port, warmup=WARMUP, duration=DURATION, rate=PORT_RATE
)

# Run-level facts only; the seven quantities live in the table of Task 4.
cards(
    card(
        "Whole run",
        [
            ("Packets offered [–]", port.cum_packet_count),
            ("Packets dropped [–]", measured["drops"]),
            ("Loss fraction [–]", f"{measured['loss']:.4f}"),
        ],
        note="Counted over the whole run, warm-up included. The M/M/1 model assumes "
        "an unbounded queue, so it only applies while nothing is dropped; with any "
        "loss the measured λ falls below the offered rate.",
    ),
    card(
        "After the warm-up",
        [
            ("Packets measured [–]", samples.delays.size),
            ("Observation window [s]", f"{DURATION - WARMUP:.0f}"),
            (
                "95th percentile of time in system [s]",
                f"{np.percentile(samples.delays, 95):.3f}",
            ),
        ],
        note="Only packets created after <code>WARMUP</code> enter the averages; a short "
        "window is the usual reason a measurement misses its theoretical value. "
        "Nineteen packets in twenty were faster than the 95th percentile: a quality "
        "requirement constrains the tail, not the mean.",
    ),
)

Before averaging anything, it is worth watching the queue work. The view below steps through 60 s of the run: press play, or drag the slider.

At $\rho = 0.4$ the server is idle most of the time, so a window picked at random would show very little. The view therefore selects the busiest 60 s after the warm-up; the run as a whole is emptier than what you see here. Pass `t0=` to look at an arbitrary stretch instead, and compare. Each box in the buffer is a waiting packet labelled with its size, the server shows the packet being transmitted and how much of it has gone, and the strip underneath marks where in the window the current instant lies.

The card beside the picture shows the two counters the tap records. Watch the moment a packet leaves the buffer and enters the server: $L_q$ drops by one while $L$ stays the same, because the packet is still in the system. That single step is the difference between the two formulas.

Nothing extra is recorded to draw this. For one port feeding a sink directly, the transmission of a packet takes `8 · size / PORT_RATE` seconds and ends when the sink receives it, so the whole history can be reconstructed from the sink log. The same reconstruction would not work across the several ports of Step 7, where only the creation time and the final arrival are known.

In [ ]:
# The same run as above, viewed one instant at a time rather than averaged.
# The window is chosen to contain a busy period; pass t0 to look elsewhere.
queue_view(sink, rate=PORT_RATE, length=60, after=WARMUP)

The two cells below show where the averages come from.

The first plots the occupancy against time. The tap samples it once per second, and 8000 samples of a quantity that jumps between 0 and a few packets cannot be read as a line, so the bars show the mean occupancy of each 100 s window instead. They scatter around the long-run mean: some windows catch a busy period, others are almost idle, and the queue view above shows what a single busy period looks like inside.

The solid line is the running mean of all samples taken so far. It starts at zero because the system starts empty, and needs a few hundred seconds to settle near its final value; that settling is the transient the warm-up period discards. On a linear axis it is squeezed into the left edge, so the lower chart repeats the running mean on a logarithmic time axis, where the first seconds, the first hundred, and the first thousand each get the same width. Increase `WARMUP` until the running mean is already flat where the shaded span ends.

In [ ]:
WINDOW = 100  # length of one averaging window [s]

running_mean = np.cumsum(samples.system) / np.arange(1, samples.times.size + 1)
window = (samples.times // WINDOW).astype(int)
window_mean = np.bincount(window, weights=samples.system) / np.bincount(window)
left = np.arange(window_mean.size) * WINDOW

# Top: mean occupancy per window, with the running mean and the long-run mean.
plot = new_plot(
    "Occupancy of the port over time", "Time [s]", "Packets in system [–]", width=800
)
plot.quad(
    left=0,
    right=WARMUP,
    bottom=0,
    top=1.1 * window_mean.max(),
    fill_color=GREY,
    fill_alpha=0.2,
    line_color=None,
    legend_label="Warm-up, excluded",
)
plot.quad(
    left=left + 0.1 * WINDOW,
    right=left + 0.9 * WINDOW,
    bottom=0,
    top=window_mean,
    fill_color=colors[0],
    fill_alpha=0.5,
    line_color=None,
    legend_label=f"Mean of each {WINDOW} s window",
)
plot.line(
    samples.times,
    running_mean,
    line_color=colors[4],
    line_width=2.5,
    legend_label="Running mean",
)
plot.line(
    [0, DURATION],
    [measured["L"], measured["L"]],
    line_color=GREY,
    line_dash="dashed",
    line_width=2,
    legend_label="Mean after warm-up",
)
legend_below(plot)

# Bottom: the running mean alone, on a logarithmic time axis for the transient.
start = samples.times > 0  # a logarithmic axis cannot show t = 0
transient = new_plot(
    "Running mean on a logarithmic time axis",
    "Time [s]",
    "Packets in system [–]",
    width=800,
    height=260,
    x_axis_type="log",
    x_range=(samples.times[start][0], DURATION),
)
transient.quad(
    left=samples.times[start][0],
    right=WARMUP,
    bottom=0,
    top=1.1 * running_mean.max(),
    fill_color=GREY,
    fill_alpha=0.2,
    line_color=None,
)
transient.line(
    samples.times[start], running_mean[start], line_color=colors[4], line_width=2.5
)
transient.line(
    [samples.times[start][0], DURATION],
    [measured["L"], measured["L"]],
    line_color=GREY,
    line_dash="dashed",
    line_width=2,
)

pn.Column(plot, transient)

The second plots the distribution of the measured time in system. The mean is only one number drawn from this distribution: the 95th percentile lies well above it, and it is the tail, not the mean, that decides whether an interactive service stays usable.

In [ ]:
p95 = np.percentile(samples.delays, 95)

plot = new_plot(
    "Distribution of the time in system",
    "Time in system [s]",
    "Probability density [s⁻¹]",
)
heights, _ = histogram(plot, samples.delays, bins=60, label="Measured")
top = 1.05 * heights.max()
plot.line(
    [measured["W"]] * 2,
    [0, top],
    line_color=colors[4],
    line_width=2,
    legend_label=f"Mean, {measured['W']:.2f} s",
)
plot.line(
    [p95] * 2,
    [0, top],
    line_color=colors[4],
    line_dash="dashed",
    line_width=2,
    legend_label=f"95th percentile, {p95:.2f} s",
)
legend_below(plot)

pn.pane.Bokeh(plot)

### Task 4 – Compare with theory

Fill in the seven theoretical values below. Each key of `theory` names one symbol, and `mm1_report` puts it in a row next to the measurement that should confirm it, the formula it follows from, and the simulation output the measurement was taken from. A value left as `np.nan` is reported as not filled in rather than compared.

1. Compute $\lambda$ and $\mu$ from the simulation parameters (`MEAN_INTERARRIVAL`, `MEAN_SIZE`, `PORT_RATE`), then the utilization $\rho$.
2. Evaluate $L$, $L_q$, $W$, and $W_q$ with the M/M/1 formulas from the README.
3. Read the Agreement column. The tolerance is 10 %, which is deliberately loose: a run of a few thousand seconds estimates these quantities to within a few per cent at best, as the spread across seeds in Step 5 shows. A row outside it is either a formula to re-check or a real limitation of the model; decide which, using the drop count and the observation window from the dashboard above.
4. Check Little's law: does $\lambda W$ reproduce the measured $L$, and $\lambda W_q$ the measured $L_q$?
5. The measured $\mu$ comes from the packets that were actually transmitted, not from `MEAN_SIZE`. Explain why the two can differ, and confirm that they agree here because nothing was dropped.

In [ ]:
theory = {
    "lambda": np.nan,  # arrival rate [1/s]
    "mu": np.nan,  # service rate [1/s]
    "rho": np.nan,  # utilization [–]
    "L": np.nan,  # mean packets in system [–]
    "L_q": np.nan,  # mean packets waiting [–]
    "W": np.nan,  # mean time in system [s]
    "W_q": np.nan,  # mean waiting time [s]
}

mm1_report(theory, measured)

# Little's law, each prediction beside the occupancy it should reproduce.
print("Little's law")
print(
    f"  λ · W   = {theory['lambda'] * measured['W']:8.4f}"
    f"     measured L   = {measured['L']:8.4f}"
)
print(
    f"  λ · W_q = {theory['lambda'] * measured['W_q']:8.4f}"
    f"     measured L_q = {measured['L_q']:8.4f}"
)

Once the table agrees, the figure below puts the same two comparisons on the plots: the theoretical $L$ against the running mean of the occupancy, and the exponential density with rate $\mu - \lambda$, which is what M/M/1 predicts for the time in system, against the measured histogram. Both theory lines are drawn from your `theory` dictionary, so they only line up when the values in it are right.

In [ ]:
# Top: the running mean against the measured and the theoretical L.
occupancy = new_plot(
    "Occupancy against the theoretical L",
    "Time [s]",
    "Packets in system [–]",
    y_range=(0, max(1.0, 2 * measured["L"])),
)
occupancy.line(
    samples.times,
    running_mean,
    line_color=colors[4],
    line_width=2.5,
    legend_label="Running mean",
)
for value, dash, color, label in (
    (measured["L"], "dashed", GREY, f"Measured L = {measured['L']:.3f}"),
    (theory["L"], "dotted", colors[5], f"Theoretical L = {theory['L']:.3f}"),
):
    occupancy.line(
        [0, DURATION],
        [value, value],
        line_color=color,
        line_dash=dash,
        line_width=3,
        legend_label=label,
    )
legend_below(occupancy)

# Bottom: the measured delays against the exponential density M/M/1 predicts.
delay = new_plot(
    "Time in system against the theoretical density",
    "Time in system [s]",
    "Probability density [s⁻¹]",
)
histogram(delay, samples.delays, bins=60, label="Measured")
t = np.linspace(0, samples.delays.max(), 300)
decay = theory["mu"] - theory["lambda"]  # rate of the predicted exponential [1/s]
delay.line(
    t,
    decay * np.exp(-decay * t),
    line_color=colors[5],
    line_dash="dashed",
    line_width=3,
    legend_label="Theory: exponential with rate μ − λ",
)
legend_below(delay)

pn.Column(occupancy, delay)

### Step 5 – Repeating the run

One run is one sample. The wiring of Step 4 is wrapped in a function below so that it can be repeated, and the table collects seeds 1, 2, and 3 against the same theoretical values. The spread column is the difference between the largest and the smallest of the three, which is the variation a single run hides.

In [ ]:
def simulate_mm1(
    mean_interarrival=MEAN_INTERARRIVAL, seed=SEED, duration=DURATION, warmup=WARMUP
):
    """
    Run the M/M/1 system of Step 4 and measure it.

    This is the wiring of Step 4 unchanged, wrapped in a function so that the
    same system can be run again with another seed or another arrival rate.

    Parameters
    ----------
    mean_interarrival : float, optional
        Mean time between arrivals [STU], by default MEAN_INTERARRIVAL.
    seed : int, optional
        Seed of the random number generator, by default SEED.
    duration : float, optional
        Total simulated time [s], by default DURATION.
    warmup : float, optional
        Initial time excluded from the measurement [s], by default WARMUP.

    Returns
    -------
    tuple[dict, MM1Samples]
        The measured quantities and the arrays behind them.
    """
    env, rng = new_run(seed)

    sink = PacketSink(env, "sink", debug=False)
    switch = Switch(
        env,
        "switch01",
        num_ports=1,
        port_capacity=BUFFER,
        port_transmission_rate=PORT_RATE,
    )
    port = switch.ports[0]
    tap = NetworkTap(env, port=port)
    source = PacketSource(
        env,
        "source",
        packet_interval=partial(rng.exponential, mean_interarrival),
        packet_size=partial(rng.exponential, MEAN_SIZE),
    )

    source.destination = port
    port.destination = sink

    env.run(until=duration)
    return mm1_measurements(
        sink, tap, port, warmup=warmup, duration=duration, rate=PORT_RATE
    )


results = {seed: simulate_mm1(seed=seed)[0] for seed in (1, 2, 3)}
seed_table(results, theory)

### Step 6 – Delay against utilization

The M/M/1 formulas say that $W = 1/(\mu - \lambda)$ grows without bound as $\rho \to 1$, however far the server still is from processing more than it can. A single run cannot show that; a sweep over several arrival rates can.

### Task 5 – Sweep the arrival rate

1. Choose mean inter-arrival times that give utilizations of roughly 0.2 to 0.95. The service rate $\mu$ does not change, so $\rho$ follows from the arrival rate alone.
2. Call `simulate_mm1(mean_interarrival=...)` for each and collect the measured $\rho$, $W$, and drop count.
3. Plot the measured mean delay as markers against the theoretical curve $W = 1/(\mu - \lambda)$ drawn as a line, with utilization on the horizontal axis.
4. The agreement degrades as $\rho \to 1$. Three effects contribute: the finite buffer begins to drop packets, the transient outlasts the warm-up period, and fewer packets complete within the run. Use the drop count and the number of measured packets to argue which one dominates at your highest utilization.

In [ ]:
mean_interarrivals = []  # values to sweep [STU]
sweep = []  # measured quantities for each of them

# for mean_interarrival in mean_interarrivals:
#     values, _ = simulate_mm1(mean_interarrival=mean_interarrival)
#     sweep.append(values)

# Plot the sweep in the style of the other figures, for example:
# plot = new_plot("Mean time in system against utilization", "Utilization ρ [–]", "Mean time in system [s]")
# plot.scatter(rho_values, w_values, size=10, color=colors[0], legend_label="Simulation")
# plot.line(rho_grid, w_theory, line_color=GREY, line_dash="dashed", line_width=2, legend_label="Theory")
# legend_below(plot)
# pn.pane.Bokeh(plot)

### Step 7 – A network of queues

Three sources, two probabilistic forks, and four switch ports form a small network. `PacketFork` sends each packet to one of its `destinations` with the given probabilities; it is given the seeded generator so that the run stays reproducible. The port rate is chosen so that a port serves 2.2 packets of mean size per second.

Every branch of every fork needs a destination. A quarter of the traffic leaving port 0 is not forwarded deeper into this switch but leaves the network, which `sink3` represents; leaving that branch unassigned would raise an error rather than silently discard the packets.

In [ ]:
env, rng = new_run()

mean_pkt_size = 100.0  # [B]
port_rate = 2.2 * 8 * mean_pkt_size  # [bit/s]

adist1 = partial(rng.exponential, 0.5)
adist2 = partial(rng.exponential, 2)
adist3 = partial(rng.exponential, 10 / 6)
sdist = partial(rng.exponential, mean_pkt_size)

sink1 = PacketSink(env, sink_id="sink1", debug=False)
sink2 = PacketSink(env, sink_id="sink2", debug=False)
sink3 = PacketSink(env, sink_id="sink3", debug=False)  # traffic leaving the network
source1 = PacketSource(env, "source1", packet_interval=adist1, packet_size=sdist)
source2 = PacketSource(env, "source2", packet_interval=adist2, packet_size=sdist)
source3 = PacketSource(env, "source3", packet_interval=adist3, packet_size=sdist)
fork1 = PacketFork(env, [0.75, 0.25], rng=rng)
fork2 = PacketFork(env, [0.65, 0.35], rng=rng)
switch = Switch(
    env,
    "switch01",
    num_ports=4,
    port_capacity=10_000,  # buffer for waiting packets [B]
    port_transmission_rate=port_rate,  # [bit/s]
)
taps = [NetworkTap(env, port=p) for p in switch.ports]  # one per port, for Task 6

# source1 → port 0 → fork1 → port 1 → fork2 → ports 2 and 3 → sinks
#                        └─ 25 % leaves the network at sink3
source1.destination = switch.ports[0]
switch.ports[0].destination = fork1
fork1.destinations[0] = switch.ports[1]
fork1.destinations[1] = sink3
switch.ports[1].destination = fork2
fork2.destinations[0] = switch.ports[2]
fork2.destinations[1] = switch.ports[3]
source3.destination = switch.ports[2]
source2.destination = switch.ports[3]
switch.ports[2].destination = sink1
switch.ports[3].destination = sink2

env.run(until=4000)

In [ ]:
network = cards(
    card(
        "Delays at the sinks",
        [
            ("Mean delay at sink 1 [s]", f"{np.mean(sink1.delays):.3f}"),
            ("Mean delay at sink 2 [s]", f"{np.mean(sink2.delays):.3f}"),
        ],
        note="Two paths are mixed at each sink: source 1 crosses ports 0, 1 and 2 (sink 1) "
        "or 0, 1 and 3 (sink 2), while source 3 enters at port 2 and source 2 at port 3. "
        "Task 6 asks why the two means differ.",
    ),
    card(
        "Traffic through the switch",
        [
            (
                "Mean packets in system, port 0 [–]",
                f"{np.mean(taps[0].system_packets):.3f}",
            ),
            ("Dropped on all ports [–]", sum(p.cum_drop_count for p in switch.ports)),
            (
                "Delivered to sinks 1 / 2 / 3 [–]",
                f"{len(sink1.delays)} / {len(sink2.delays)} / {len(sink3.delays)}",
            ),
        ],
        note="Port 0 carries source 1 alone and is the busiest; the mean includes the "
        "warm-up. Divide the delivered counts by the 4000 s run to get the arrival rate "
        "at each sink and check them against Task 6. Sink 3 receives the quarter of "
        "port 0 traffic that leaves the network at fork 1.",
    ),
)

plot = new_plot(
    "Departure intervals at the two sinks",
    "Time between arrivals [s]",
    "Frequency of occurrence [–]",
)
bins = np.linspace(0, 4, 60)
histogram(
    plot,
    sink1.interarrivals,
    bins=bins,
    density=False,
    color=colors[0],
    alpha=0.5,
    label="Sink 1",
)
histogram(
    plot,
    sink2.interarrivals,
    bins=bins,
    density=False,
    color=colors[4],
    alpha=0.5,
    label="Sink 2",
)
legend_below(plot)

pn.Column(network, pn.pane.Bokeh(plot))

> [!NOTE]
> The histogram above shows the intervals between *departures* at each sink, not the intervals generated by the sources. For a stable M/M/1 queue the departures again form a Poisson process, so the shape is exponential once more — a result known as Burke's theorem.

### Task 6 – Offered traffic per port

Every port is tapped, so the same theory-against-measurement table applies here, one group of rows per port.

1. Work out the arrival rate $\lambda$ at each of the four ports from the source rates and the fork probabilities, and the offered traffic $A = \lambda S$ of each port. The mean service time $S$ follows from `mean_pkt_size` and `port_rate`.
2. Fill them into `port_theory` below and read the Agreement column.
3. Check your rates against the packet counts at the three sinks reported above.
4. The measured $\rho$ has no theoretical column, because it is the quantity your $A$ is supposed to predict. Where the two differ, say whether loss or the length of the run explains it.
5. Explain the difference between the delays observed at sinks 1 and 2.

In [ ]:
DURATION_NETWORK = 4000  # [s], the run above

port_theory = {
    0: {"lambda": np.nan, "A": np.nan},
    1: {"lambda": np.nan, "A": np.nan},
    2: {"lambda": np.nan, "A": np.nan},
    3: {"lambda": np.nan, "A": np.nan},
}

port_report(
    port_theory,
    port_measurements(
        switch.ports, taps, duration=DURATION_NETWORK, mean_size=mean_pkt_size
    ),
)

## Questions

1. Fifteen customers per hour arrive at a payphone according to a Poisson process, and a call lasts on average 3 minutes with an exponential distribution. Is a second payphone needed if the mean waiting time is not to exceed 3 minutes?
2. In Step 4, what are $\lambda$, $\mu$, and $\rho$, and which formula should the measured mean time in system match: $W$ or $W_q$?
3. What does Little's law predict for the mean occupancy in Step 4, and does the network tap confirm it?
4. Why does the mean delay of an M/M/1 system increase sharply as $\rho$ approaches 1, even though the server is still not saturated?
5. Which of the three distributions in the theory section is memoryless, and why does that matter for the choice of a traffic model?
6. The tap in Step 4 reports a mean `system_packets` and a mean `queue_packets` that differ by roughly $\rho$. Explain why, without using the M/M/1 formulas.